In [ ]:
import os, eda_analysis

cfg = eda_analysis.EdaConfig(family="compute/cost")
S   = eda_analysis.notebook_setup(cfg)

# `compute/cost` -- the spend axis

**Why this family exists.** Every other family in this EDA is indexed by ITERATION, and an
iteration is not a fixed unit of spend:

* a K=5 step pays for K extra simulated turns per candidate before the grader ever sees it, so it
  costs more than a K=0 step;
* PTO bills a **preference-build** phase that GRPO does not have at all (GRPO computes its reward
  inside the training loop), so one PTO iteration and one GRPO iteration are different purchases.

Two arms compared at the same iteration are therefore **not** compared at the same price, and a
lever's sign can differ between the two readings. This family supplies the second reading.

**Where the numbers come from.** `data.load_timing` reads the trainer's own append-only per-phase
log, `runs/<ARM>/iteration_<N>/timing_sessions.jsonl`, and sums it over every session that worked
on the iteration. There is **no mtime reconstruction anywhere in Exp4** -- Exp3 needed 1,336 lines
of it because its per-process fields undercounted a resumed iteration by nearly 2x, and the fix
was to log properly rather than to reconstruct better. Two consequences follow, and both are
surfaced below:

* **`n_sessions_production > 1` means the iteration was RESUMED.** The cumulative log is still
  correct (that is the point of appending), but any per-PROCESS field elsewhere --
  `iteration_metadata.json`'s own timings -- is an undercount for that iteration. Never mix the
  two sources in one table. The *production* session count is the one to read: the post-loop
  final-eval pass appends an `eval_gen_s`-only session to the last training iteration of every
  healthy arm, so a raw `n_sessions > 1` would report every completed arm as interrupted.
* **`n_sessions == 0` means nothing was logged.** That cost is unrecoverable, not free. Such rows
  are flagged rather than silently summed as zero.

**No API axis.** Exp4's default stack serves every role from one local vLLM server, so the API
bill is $0 by construction and GPU-hours are the whole cost. If a role is ever flipped to a vendor
API, that stops being true and a second axis has to be added here.

**What "cost to reach state N" means.** Iteration `n` trains the adapter that IS model state `n`,
so a state's cost is the sum of iterations `1..n`. The conversations *for* state `n` are generated
at the start of iteration `n+1`, so this bills the cost of PRODUCING the policy and not the cost
of measuring it. State 0 (the untrained base) is free.

That rule is why the cumulative sum is over **`production_s`** (generate + build + train) and not
`total_s`. The final policy has no iteration `n+1` to be measured in, so its generate-only eval
pass is logged as `eval_gen_s` against the LAST training iteration -- and billing `total_s` would
price exactly one point per arm, the endpoint every budget sweep is read at, under a different
rule from all the others and shifted right by a whole generation pass. Two arms whose final-eval
passes cost different amounts would then be compared at budgets that mean different things.

**Read the budget sweep, not one endpoint row.** Which arm is ahead is a function of budget: an
arm can trail badly at a small budget and draw level later, and a fixed endpoint can freeze an arm
*after* its regression -- in Exp3 the sweep and the fixed endpoint disagreed at the top budget for
exactly that reason. The sweep reports each arm's **best checkpoint within the budget** and, next
to it, its **last checkpoint within the budget**, so the disagreement is visible instead of being
a choice made silently.

In [ ]:
# ---------------------------------------------------------------------------
# Timing log + scores. Everything here degrades to an explicit "no data yet"
# artifact: an arm can be trained but unscored (timing without scores), scored but
# with no timing log, or neither.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from eda_analysis import constants, data, exports, plotting, stats

SEED = S.CFG.boot_seed
FOCUS = S.CFG.focus_metric

# The per-phase wall-clock log, summed over sessions. One row per (arm, TRAINING iteration).
T = data.load_timing(S.ARMS)
# The trainer's phase keys, not every column ending in `_s`: the frame also carries the derived
# roll-ups `total_s` (every phase) and `production_s` (every phase EXCEPT eval-gen), and a
# heuristic that swept those in would double-count them in every share and every stacked bar.
PHASES = [c for c in data.PHASE_KEYS if c in T.columns]
PHASE_LABEL = {"generation_s": "generate", "pref_pair_s": "build (PTO only)",
               "training_s": "train", "eval_gen_s": "eval-gen"}
# The roll-up row's label, named ONCE: the multiplier tables write it and the numbers ledger
# filters on it, and a literal in each drifted silently -- the ledger matched zero rows and the
# headline `cost.k_multiplier.*` keys were simply absent from compute_cost.json, with nothing
# in the render reporting the omission.
TOTAL_PHASE_LABEL = "TOTAL (production)"

ALL = data.scores_by_judge(S.ARMS, rep=S.CFG.judge_rep, attach_persona=False)
if ALL.empty and not S.SCORES.empty:
    ALL = S.SCORES
HAVE_SCORES = not ALL.empty
HAVE_TIMING = not T.empty

JUDGES = sorted(ALL["judge"].unique()) if HAVE_SCORES else [S.JUDGE]
ARMS = sorted(T["arm_label"].unique()) if HAVE_TIMING else []
PALETTE = plotting.arm_palette(ARMS) if ARMS else {}

NO_TIMING = ("NO TIMING YET -- no runs/<ARM>/iteration_<N>/timing_sessions.jsonl on disk. The "
             "trainer writes it per phase; nothing here can be reconstructed after the fact.")
NO_SCORES = ("NO SCORES YET -- the spend axis exists but nothing has been graded, so cost cannot "
             "be put against outcome. Run notebooks/scoring/Run_Eval.ipynb and re-render.")

exports.reset_results()
exports.save_provenance(S.CFG, ALL)


def placeholder(name, message=NO_TIMING, group=None, caption=None):
    fig = plt.figure(figsize=(7.6, 1.9))
    ax = fig.add_subplot(111)
    ax.axis("off")
    ax.text(0.5, 0.5, message, ha="center", va="center", fontsize=8.5,
            color="#777777", wrap=True)
    exports.save_fig(fig, name, group=group, caption=caption or message)
    plt.close(fig)


def as_arm(df):
    # Every frame from data.py carries `arm` as a DUPLICATE of `arm_label` (one fact, two
    # spellings, because config keys on one and the figure builders default to the other).
    # Renaming arm_label -> arm without dropping the duplicate first produces two columns
    # called `arm`, and the next sort_values on it raises "label is not unique".
    return df.drop(columns=["arm"], errors="ignore").rename(columns={"arm_label": "arm"})


print(f"timing rows: {len(T)}  arms: {ARMS or '(none)'}")
print(f"phases     : {PHASES}")
print(f"judges     : {JUDGES if HAVE_SCORES else '(nothing scored)'}")
if not HAVE_TIMING:
    print(NO_TIMING)
if not HAVE_SCORES:
    print(NO_SCORES)

## 1. GPU-hours per arm, and the phase breakdown

`build` is PTO-only: GRPO computes its reward inside the training loop, so it has no
preference-build phase. In Exp3 that phase was PTO's *dominant* one, which is why a per-step
timing comparison between the two methods is not a comparison of what they cost.

`resumed_iterations` counts iterations with more than one logged session, and
`unlogged_iterations` counts iterations that logged nothing -- an unrecoverable cost, and the one
number in this family that is wrong in a way no later analysis can fix.

In [ ]:
def per_arm(t):
    # Total hours per arm, split by phase, with the two integrity counters.
    if t.empty:
        return pd.DataFrame()
    agg = {p: (p, "sum") for p in PHASES}
    out = (t.groupby(["arm_label", "method", "k", "mode", "mcl"], as_index=False)
             .agg(iterations=("iteration", "nunique"),
                  resumed_iterations=("resumed", "sum"),
                  sessions=("n_sessions", "sum"),
                  total_s=("total_s", "sum"),
                  production_s=("production_s", "sum"),
                  **agg))
    out["unlogged_iterations"] = [
        int((t[(t["arm_label"] == a)]["n_sessions"] == 0).sum()) for a in out["arm_label"]]
    out["logged_iterations"] = out["iterations"] - out["unlogged_iterations"]
    out["gpu_hours"] = out["total_s"] / 3600.0
    # The cost axis: everything except the generate-only eval pass, which MEASURES the final
    # policy rather than producing it. cumulative_cost and the budget sweep bill this one.
    out["production_h"] = out["production_s"] / 3600.0
    for p in PHASES:
        out[f"{p[:-2]}_h"] = out[p] / 3600.0
        out[f"{p[:-2]}_share"] = np.where(out["total_s"] > 0, out[p] / out["total_s"], np.nan)
    # Divided by the LOGGED iterations, not by all of them: an unlogged iteration contributes
    # nothing to the numerator, so counting it in the denominator would silently deflate the
    # per-iteration cost of exactly the arms with a telemetry gap.
    out["h_per_logged_iteration"] = np.where(out["logged_iterations"] > 0,
                                             out["gpu_hours"] / out["logged_iterations"], np.nan)
    keep = (["arm_label", "method", "k", "mode", "mcl", "iterations", "logged_iterations",
             "gpu_hours", "production_h", "h_per_logged_iteration"]
            + [f"{p[:-2]}_h" for p in PHASES] + [f"{p[:-2]}_share" for p in PHASES]
            + ["resumed_iterations", "unlogged_iterations", "sessions"])
    return (out[keep].rename(columns={"arm_label": "arm"})
                     .sort_values(["method", "k", "arm"]).reset_index(drop=True))


BY_ARM = per_arm(T)
exports.save_table(
    BY_ARM, "compute_by_arm",
    caption=("GPU-hours per arm from the trainer's append-only per-phase log "
             "(timing_sessions.jsonl), summed over every session. `build` is PTO-only -- GRPO "
             "computes its reward inside the training loop and has no preference-build phase. "
             "`gpu_hours` is the FULL bill; `production_h` excludes `eval_gen` -- the "
             "generate-only pass that measures the final policy rather than producing it -- and "
             "is the axis cumulative_cost and the budget sweep are billed on. "
             "`resumed_iterations` > 0 means some iteration did PRODUCTION work in several "
             "sessions (the cumulative sum here is still correct; per-process fields elsewhere "
             "are not; a final-eval session on its own is not a resume). "
             "`unlogged_iterations` > 0 means a cost that cannot be recovered at all, so "
             "`gpu_hours` for that arm is an UNDERCOUNT and the per-iteration figure divides by "
             "the logged iterations only."))

BY_ITER = as_arm(T.copy())
if not BY_ITER.empty:
    BY_ITER["gpu_hours"] = BY_ITER["total_s"] / 3600.0
    for p in PHASES:
        BY_ITER[f"{p[:-2]}_h"] = BY_ITER[p] / 3600.0
    BY_ITER = BY_ITER[["arm", "method", "k", "iteration", "state_index", "gpu_hours"]
                      + [f"{p[:-2]}_h" for p in PHASES]
                      + ["n_sessions", "n_sessions_production", "resumed"]]
exports.save_table(
    BY_ITER, "compute_by_iteration",
    caption=("One row per (arm, TRAINING iteration): hours by phase, summed over sessions. "
             "`iteration` is the training pass; `state_index = iteration - 1` is the model state "
             "it trained FROM. Join to a score frame on state_index, never on iteration. "
             "`resumed` (n_sessions_production > 1) marks iterations whose per-process timings "
             "elsewhere undercount -- the final-eval session that every arm's LAST iteration "
             "carries is not a resume; n_sessions == 0 means nothing was logged and the cost is "
             "lost."))

if BY_ARM.empty:
    print(NO_TIMING)
else:
    print(f"{len(BY_ARM)} arm(s), {int(BY_ARM['iterations'].sum())} logged iteration(s), "
          f"{BY_ARM['gpu_hours'].sum():.2f} GPU-h total")
    display(BY_ARM)

## 2. Integrity -- resumed and unlogged iterations

These are listed explicitly rather than folded into a footnote, because they are the two ways this
family's numbers can mislead: a resumed iteration invites someone to quote a per-process figure
that is an undercount, and an unlogged iteration silently bills as free.

In [ ]:
flags = pd.DataFrame()
if not T.empty:
    flagged = T[(T["resumed"]) | (T["n_sessions"] == 0)].copy()
    if not flagged.empty:
        flagged["gpu_hours"] = flagged["total_s"] / 3600.0
        flagged["issue"] = np.where(flagged["n_sessions"] == 0, "UNLOGGED (cost lost)",
                                    "RESUMED (cumulative log is the correct number)")
        flags = (as_arm(flagged)
                 [["arm", "method", "k", "iteration", "n_sessions",
                   "n_sessions_production", "gpu_hours", "issue"]]
                 .sort_values(["arm", "iteration"]).reset_index(drop=True))

exports.save_table(
    flags, "timing_integrity",
    caption=("Iterations that need a caveat. RESUMED (n_sessions_production > 1): the per-phase "
             "sum used everywhere in this family is correct, but any per-PROCESS timing for that "
             "iteration is an undercount -- this is the Exp3 defect that made a 7.7 h iteration "
             "report 14,501 s. The post-loop final-eval pass appends an eval-gen-only session "
             "to each arm's LAST iteration and is deliberately not counted as a resume. "
             "UNLOGGED (n_sessions == 0): nothing was recorded, so the cost is "
             "unrecoverable and every total below is short by it. An empty table is the good "
             "case."))
if flags.empty:
    print("no resumed or unlogged iterations" if not T.empty else NO_TIMING)
else:
    display(flags)

## 3. Step multipliers -- what K costs, and what the method costs

Per-ITERATION medians, taken over the iterations each arm actually logged. Medians rather than
means because a single resumed or straggling iteration should not set the ratio.

* `k_step_multiplier` holds the optimizer fixed and varies K: the cost of look-ahead, per phase
  and in total. Look-ahead extends every candidate by K simulated turns before grading, so the
  ratio is expected to be well above 1 and to sit mostly in the phases that generate.
* `method_step_multiplier` holds K fixed and varies the method (`PTO / GRPO`, alphabetical). The
  `build` row is where PTO's preference construction shows up; GRPO's is structurally zero, so
  that ratio is undefined rather than infinite.

In [ ]:
def multiplier(t, hold, vary):
    # Median per-ITERATION hours at the highest level of `vary` over the lowest, holding `hold`.
    if t.empty:
        return pd.DataFrame()
    rows = []
    for key, g in t.groupby(hold, sort=True):
        levels = sorted(g[vary].unique())
        if len(levels) < 2:
            continue
        lo, hi = levels[0], levels[-1]
        g_lo, g_hi = g[g[vary] == lo], g[g[vary] == hi]
        # production_s, not total_s: only the LAST iteration of each arm carries the final-eval
        # pass, so a total-based median would inflate whichever arm's median lands on it and the
        # ratio would stop being a per-step cost.
        for col in ["production_s"] + PHASES:
            m_lo = float(g_lo[col].median()) / 3600.0
            m_hi = float(g_hi[col].median()) / 3600.0
            rows.append({"held": f"{hold}={key}", "varied": vary,
                         "level_lo": lo, "level_hi": hi,
                         "phase": (TOTAL_PHASE_LABEL if col == "production_s"
                                   else PHASE_LABEL.get(col, col)),
                         "median_h_lo": m_lo, "median_h_hi": m_hi,
                         "multiplier": (m_hi / m_lo) if m_lo > 0 else np.nan,
                         "n_iter_lo": int(len(g_lo)), "n_iter_hi": int(len(g_hi))})
    return pd.DataFrame(rows)


K_MULT = multiplier(T, "method", "k")
METHOD_MULT = multiplier(T, "k", "method")

exports.save_table(
    K_MULT, "k_step_multiplier",
    caption=("Cost of look-ahead per TRAINING ITERATION, optimizer held fixed: median hours at "
             "the arm's highest K over median hours at its lowest, per phase and in total. The "
             "TOTAL row is PRODUCTION hours (generate + build + train); eval-gen keeps its own "
             "row because only each arm's last iteration carries it. "
             "Medians over the iterations actually logged (n_iter_* columns), so one resumed or "
             "straggling iteration cannot set the ratio. A multiplier of 1.9 means a K=5 "
             "iteration cost 1.9x a K=0 one for that optimizer."))
exports.save_table(
    METHOD_MULT, "method_step_multiplier",
    caption=("Cost of the METHOD per training iteration, K held fixed: median hours for PTO over "
             "median hours for GRPO (levels are alphabetical, so the ratio is PTO / GRPO). The "
             "`build` row is PTO's preference construction, which GRPO does not have -- its "
             "denominator is structurally zero, so that multiplier is undefined, not infinite. "
             "This is the cost side of RQ-ii: method/contrast matches on iteration, not on price."))
if K_MULT.empty and METHOD_MULT.empty:
    print(NO_TIMING if T.empty else "only one level on disk for both K and method -- no ratio yet")
else:
    display(K_MULT)
    display(METHOD_MULT)

## 4. Cumulative spend per model state

The cost of *producing* each model state: state `N` is the adapter that iteration `N` wrote, so it
is billed the sum of iterations `1..N`, and state 0 (the untrained base) is free. Conversations
for state `N` are generated at the start of iteration `N+1`, so the measurement cost of a state is
attributed to the next iteration, not to the state itself.

The sum is over `production_s` (generate + build + train), **not** `total_s`. The final policy has
no iteration `N+1`, so its generate-only eval pass is logged as `eval_gen_s` against the last
training iteration -- billing `total_s` would charge exactly one state per arm, the endpoint every
budget sweep is read at, for its own measurement.

This table is what turns every iteration-indexed result elsewhere into a price.

In [ ]:
def cumulative_cost(t):
    # Cost to REACH each model state: state N = the adapter iteration N wrote, so it is billed
    # iterations 1..N. State 0 (the untrained base) is free.
    if t.empty:
        return pd.DataFrame(columns=["arm_label", "method", "k", "state", "gpu_hours",
                                     "iterations_billed", "resumed_so_far"])
    rows = []
    for arm, g in t.groupby("arm_label", sort=True):
        g = g.sort_values("iteration")
        method, k = str(g["method"].iloc[0]), int(g["k"].iloc[0])
        rows.append({"arm_label": arm, "method": method, "k": k, "state": 0,
                     "gpu_hours": 0.0, "iterations_billed": 0, "resumed_so_far": 0})
        total, resumed = 0.0, 0
        for _, r in g.iterrows():
            # production only -- see the section note: total_s would charge each arm's LAST
            # state for generating its own eval conversations, which no other state pays for.
            total += float(r["production_s"]) / 3600.0
            resumed += int(bool(r["resumed"]))
            rows.append({"arm_label": arm, "method": method, "k": k,
                         "state": int(r["iteration"]), "gpu_hours": total,
                         "iterations_billed": int(r["iteration"]), "resumed_so_far": resumed})
    return pd.DataFrame(rows)


COST = cumulative_cost(T)
exports.save_table(
    as_arm(COST), "cumulative_cost",
    caption=("Cumulative GPU-hours to PRODUCE each model state (state N is billed training "
             "iterations 1..N; state 0, the untrained base, is free). Summed over "
             "`production_s` = generate + build + train: the conversations that MEASURE state N "
             "are generated in iteration N+1, and the FINAL state's eval pass -- which has no "
             "iteration N+1 and is logged as `eval_gen_s` against the last training iteration -- "
             "is excluded for the same reason, so every state is priced under one rule. "
             "`resumed_so_far` counts how many of the billed iterations did production work in "
             "more than one session."))
if COST.empty:
    print(NO_TIMING)
else:
    display(as_arm(COST))

## 5. Where the hours went

Two views of the same log: the cumulative spend each arm needed to reach a given model state, and
the share of that spend by phase. `build` appears only for PTO arms.

In [ ]:
if COST.empty:
    placeholder("compute_trajectory", caption="No timing log to plot. " + NO_TIMING)
else:
    fig = plt.figure(figsize=(7.4, 4.4))
    ax = fig.add_subplot(111)
    for arm, g in COST.groupby("arm_label", sort=True):
        g = g.sort_values("state")
        ax.plot(g["state"], g["gpu_hours"], marker="o", lw=1.6,
                color=PALETTE.get(arm, "#555555"), label=arm)
    ax.set_xlabel("model state (adapter produced by training iteration N)")
    ax.set_ylabel("cumulative GPU-hours")
    ax.set_title("Cumulative spend to reach each model state")
    ax.legend(title="arm", fontsize=8, frameon=False, loc="best")
    fig.tight_layout()
    exports.save_fig(
        fig, "compute_trajectory",
        caption=("Cumulative GPU-hours (trainer session log, summed over sessions) needed to "
                 "produce each model state, per arm. A steeper line is a more expensive "
                 "iteration; the same x position on two lines is the same iteration index but "
                 "NOT the same price, which is the whole reason this family exists."))
    plt.close(fig)

if BY_ARM.empty:
    placeholder("cost_breakdown", caption="No timing log to plot. " + NO_TIMING)
else:
    fig = plt.figure(figsize=(7.4, 4.2))
    ax = fig.add_subplot(111)
    arms = list(BY_ARM["arm"])
    bottom = np.zeros(len(arms))
    shades = ["#4C72B0", "#DD8452", "#55A868", "#8172B3", "#937860"]
    for i, p in enumerate(PHASES):
        vals = BY_ARM[f"{p[:-2]}_h"].to_numpy(dtype=float)
        ax.bar(arms, vals, bottom=bottom, label=PHASE_LABEL.get(p, p),
               color=shades[i % len(shades)], edgecolor="white", linewidth=0.6)
        bottom = bottom + np.nan_to_num(vals)
    ax.set_ylabel("GPU-hours")
    ax.set_title("Where each arm's hours went")
    ax.legend(title="phase", fontsize=8, frameon=False)
    ax.tick_params(axis="x", rotation=20)
    fig.tight_layout()
    exports.save_fig(
        fig, "cost_breakdown",
        caption=("Total GPU-hours per arm split by trainer phase. `build` is PTO's preference "
                 "construction and is structurally absent for GRPO, which computes its reward "
                 "inside the training loop -- so a per-step comparison between the two methods "
                 "is not a comparison of what they cost."))
    plt.close(fig)
print("spend figures rendered")

## 6. Score at matched budget -- the budget sweep

For every budget on the grid (the grid is the set of cumulative costs any arm actually reached),
each arm reports:

* `best_*` -- its **best** checkpoint costing no more than the budget, chosen on
  `sign_of(metric) * score` so MICI is ranked the right way round;
* `last_*` -- its **latest** checkpoint within the budget, i.e. what a fixed-endpoint reading
  would have quoted;
* `frozen_after_peak = True` when those two disagree, which is exactly the case where a
  fixed-endpoint reading freezes an arm *after* its regression and reports the wrong winner.

The verdict table names the leader under both rules and flags where they disagree. **Quote the
sweep, not a single row**: the leader is a function of budget, and picking one budget picks the
answer.

Both graders are reported, in separate artifacts named after the grader (there is no `<judge>/`
directory level in Exp4). Their scores are never averaged.

In [ ]:
def state_means(df, judge, metric):
    empty = pd.DataFrame({"arm_label": pd.Series(dtype="object"),
                          "state": pd.Series(dtype="int64"),
                          "score": pd.Series(dtype="float64"),
                          "n": pd.Series(dtype="int64")})
    if df.empty:
        return empty
    g = df[(df["judge"] == judge) & (df["metric"] == metric)]
    if g.empty:
        return empty
    out = (g.groupby(["arm_label", "iteration"], as_index=False)
             .agg(score=("score", "mean"), n=("score", "count")))
    return out.rename(columns={"iteration": "state"})


def budget_sweep(points, metric):
    # Each arm's BEST and LAST checkpoint within each budget on the grid. The grid is the set of
    # cumulative costs any arm actually reached, so every crossing point is represented exactly.
    if points.empty:
        return pd.DataFrame()
    usable = points.dropna(subset=["score"])
    if usable.empty:
        return pd.DataFrame()
    sign = int(constants.sign_of(metric))
    budgets = sorted({round(float(b), 6) for b in usable["gpu_hours"]})
    rows = []
    for budget in budgets:
        for arm, g in usable.groupby("arm_label", sort=True):
            elig = g[g["gpu_hours"] <= budget + 1e-9]
            if elig.empty:
                continue
            oriented = sign * elig["score"].to_numpy(dtype=float)
            best = elig.iloc[int(np.argmax(oriented))]
            last = elig.sort_values("gpu_hours", kind="mergesort").iloc[-1]
            rows.append({
                "budget_gpu_h": budget, "arm": arm, "metric": metric,
                "best_state": int(best["state"]), "best_score": float(best["score"]),
                "best_cost_gpu_h": float(best["gpu_hours"]),
                "last_state": int(last["state"]), "last_score": float(last["score"]),
                "last_cost_gpu_h": float(last["gpu_hours"]),
                "frozen_after_peak": int(best["state"]) != int(last["state"]),
                "n_personas": int(best["n"]), "sign": sign})
    return pd.DataFrame(rows)


def budget_verdicts(sweep):
    # Who leads at each budget, under the best-within-budget rule and under the
    # fixed-endpoint rule, and whether the two agree.
    if sweep.empty:
        return pd.DataFrame()
    rows = []
    for budget, g in sweep.groupby("budget_gpu_h", sort=True):
        sign = int(g["sign"].iloc[0])
        by_best = g.assign(o=sign * g["best_score"]).sort_values("o", ascending=False)
        by_last = g.assign(o=sign * g["last_score"]).sort_values("o", ascending=False)
        top = by_best.iloc[0]
        margin = (float(top["best_score"] * sign - by_best.iloc[1]["best_score"] * sign)
                  if len(by_best) > 1 else np.nan)
        rows.append({
            "budget_gpu_h": float(budget), "arms_within_budget": int(len(g)),
            "leader_best_within_budget": str(top["arm"]),
            "leader_best_state": int(top["best_state"]),
            "leader_best_score": float(top["best_score"]),
            "margin_over_runner_up": margin,
            "runner_up": str(by_best.iloc[1]["arm"]) if len(by_best) > 1 else "",
            "leader_fixed_endpoint": str(by_last.iloc[0]["arm"]),
            "rules_agree": str(top["arm"]) == str(by_last.iloc[0]["arm"]),
            "leader_frozen_after_peak": bool(top["frozen_after_peak"])})
    return pd.DataFrame(rows)


SWEEPS = {}
for judge in JUDGES:
    means = state_means(ALL, judge, FOCUS)
    points = (COST.merge(means, on=["arm_label", "state"], how="left")
              if not COST.empty else pd.DataFrame())
    sweep = budget_sweep(points, FOCUS) if not points.empty else pd.DataFrame()
    verdicts = budget_verdicts(sweep)
    SWEEPS[judge] = (points, sweep, verdicts)

    exports.save_table(
        sweep, f"budget_sweep_{judge}",
        caption=(f"Score at matched BUDGET on {FOCUS}, grader {judge}. For each budget (the grid "
                 f"is every cumulative cost an arm actually reached) and each arm: its BEST "
                 f"checkpoint within that budget, chosen on sign_of(metric) * score, and its LAST "
                 f"checkpoint within it -- what a fixed-endpoint reading would quote. "
                 f"`frozen_after_peak` marks where those disagree. Scores are per-state means "
                 f"over the personas scored ({FOCUS}); cost is cumulative GPU-hours to produce "
                 f"the state."))
    exports.save_table(
        verdicts, f"budget_verdicts_{judge}",
        caption=(f"Who leads on {FOCUS} at each budget under grader {judge}, by the "
                 f"best-within-budget rule and by the fixed-endpoint rule, with `rules_agree` "
                 f"marking the budgets where the two disagree -- which is where quoting a single "
                 f"endpoint would report a different winner. Never averaged across graders."))

if not HAVE_TIMING:
    print(NO_TIMING)
elif not HAVE_SCORES:
    print(NO_SCORES)
else:
    for judge, (_p, sweep, verdicts) in SWEEPS.items():
        print(f"{judge}: {len(sweep)} sweep rows, {len(verdicts)} budgets")
        if not verdicts.empty:
            display(verdicts.tail(8))

## 7. Score against spend

The same points as a curve: each arm's mean score on the training-reward axis against the
cumulative GPU-hours that bought it, annotated with the model state. This is the only view in the
EDA indexed by price rather than by iteration.

Read the lever off the whole curve. Which arm is ahead depends on where you stop, so a crossing is
information and a single quoted pair is a choice.

In [ ]:
for judge in JUDGES:
    points, sweep, _v = SWEEPS.get(judge, (pd.DataFrame(), pd.DataFrame(), pd.DataFrame()))
    usable = points.dropna(subset=["score"]) if not points.empty else points
    name = f"cost_benefit_{judge}"
    if usable is None or usable.empty:
        placeholder(name, message=(NO_TIMING if not HAVE_TIMING else NO_SCORES),
                    caption=f"No cost/score points for grader {judge}.")
    else:
        fig = plotting.cost_benefit(
            usable, x="gpu_hours", y="score", arm_col="arm_label", label_col="state",
            palette=PALETTE, title=f"{constants.short_label(FOCUS)} vs spend ({judge})",
            xlabel="cumulative GPU-hours to produce the state",
            ylabel=f"mean {FOCUS} across personas")
        exports.save_fig(
            fig, name,
            caption=(f"Mean {FOCUS} (across the personas scored) against the cumulative GPU-hours "
                     f"that produced each model state, grader {judge}; markers are annotated with "
                     f"the state index. Cost comes from the trainer's session log, never from "
                     f"file mtimes. Read the lever off the whole curve: which arm is ahead is a "
                     f"function of budget."))
        plt.close(fig)

    bname = f"budget_sweep_{judge}"
    if sweep is None or sweep.empty:
        placeholder(bname, message=(NO_TIMING if not HAVE_TIMING else NO_SCORES),
                    caption=f"No budget sweep for grader {judge}.")
        continue
    fig = plt.figure(figsize=(7.4, 4.4))
    ax = fig.add_subplot(111)
    for arm, g in sweep.groupby("arm", sort=True):
        g = g.sort_values("budget_gpu_h")
        color = PALETTE.get(arm, "#555555")
        ax.step(g["budget_gpu_h"], g["best_score"], where="post", lw=1.8, color=color,
                label=f"{arm} (best within budget)")
        ax.step(g["budget_gpu_h"], g["last_score"], where="post", lw=1.0, ls="--", alpha=0.75,
                color=color, label=f"{arm} (fixed endpoint)")
    ax.set_xlabel("budget (cumulative GPU-hours)")
    ax.set_ylabel(f"mean {FOCUS}")
    ax.set_title(f"Score at matched budget ({judge})")
    ax.legend(fontsize=7, frameon=False, loc="best")
    fig.tight_layout()
    exports.save_fig(
        fig, bname,
        caption=(f"Budget sweep on {FOCUS}, grader {judge}: solid = each arm's BEST checkpoint "
                 f"within the budget (ranked on sign_of(metric) * score), dashed = its LAST "
                 f"checkpoint within the budget, which is what a fixed-endpoint reading quotes. "
                 f"Where the two separate, the arm peaked and then regressed, and the endpoint "
                 f"reading reports the wrong number."))
    plt.close(fig)
print("cost/benefit figures rendered")

## 8. Number ledger and index

The citable spend numbers: total hours per arm, the step multipliers with their arithmetic shown,
and the leader at the largest budget under both rules.

In [ ]:
values = {
    "cost.source": {"value": "runs/<ARM>/iteration_<N>/timing_sessions.jsonl",
                    "source": "tables/compute_by_iteration.md",
                    "note": ("the trainer's append-only per-phase log, summed over sessions. "
                             "Exp4 does NO mtime reconstruction.")},
    "cost.api_usd": {"value": 0.0, "source": "",
                     "note": ("the default stack serves every role from one local vLLM server; "
                              "GPU-hours are the whole cost")},
    "cost.state_billing": {"value": "state N = sum(iterations 1..N)", "source": "",
                           "note": "cost of PRODUCING the policy; measurement is billed to N+1"},
}
if not BY_ARM.empty:
    values["cost.total_gpu_hours"] = {
        "value": float(BY_ARM["gpu_hours"].sum()), "source": "tables/compute_by_arm.md",
        "note": " + ".join(f"{a}={h:.2f}" for a, h in
                           zip(BY_ARM["arm"], BY_ARM["gpu_hours"]))}
    for _, r in BY_ARM.iterrows():
        values[f"cost.arm.{r['arm']}.gpu_hours"] = {
            "value": float(r["gpu_hours"]), "source": "tables/compute_by_arm.md",
            "note": (f"{int(r['logged_iterations'])} logged of {int(r['iterations'])} "
                     f"iteration(s), {float(r['h_per_logged_iteration']):.2f} h per logged "
                     f"iteration, {int(r['resumed_iterations'])} resumed, "
                     f"{int(r['unlogged_iterations'])} unlogged")}
for label, table in (("k", K_MULT), ("method", METHOD_MULT)):
    if table.empty:
        continue
    for _, r in table[table["phase"] == TOTAL_PHASE_LABEL].iterrows():
        values[f"cost.{label}_multiplier.{r['held']}"] = {
            "value": float(r["multiplier"]), "source": f"tables/{label}_step_multiplier.md",
            "note": (f"median h/iteration {float(r['median_h_hi']):.3f} "
                     f"({r['varied']}={r['level_hi']}, n={int(r['n_iter_hi'])}) / "
                     f"{float(r['median_h_lo']):.3f} "
                     f"({r['varied']}={r['level_lo']}, n={int(r['n_iter_lo'])})")}
for judge, (_p, _s, verdicts) in SWEEPS.items():
    if verdicts is None or verdicts.empty:
        continue
    top = verdicts.sort_values("budget_gpu_h").iloc[-1]
    values[f"cost.{judge}.leader_at_max_budget"] = {
        "value": str(top["leader_best_within_budget"]),
        "source": f"tables/budget_verdicts_{judge}.md",
        "note": (f"at {float(top['budget_gpu_h']):.2f} GPU-h, best-within-budget rule "
                 f"(state {int(top['leader_best_state'])}, {FOCUS}="
                 f"{float(top['leader_best_score']):.3f}); fixed-endpoint rule says "
                 f"{top['leader_fixed_endpoint']}, rules_agree={bool(top['rules_agree'])}")}

exports.save_numbers(
    "compute_cost", values,
    caption=("Citable spend numbers. Every composite shows its arithmetic. Budget leaders are "
             "per grader and reported under both the best-within-budget and the fixed-endpoint "
             "rule, because the two can disagree."))
print(exports.build_index())